# CRACK SEGMENTATION 

# Import libraries

In [ ]:
# Cell 2: Import libraries
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from PIL import Image, ImageOps
from collections import Counter, defaultdict
import yaml
import warnings
warnings.filterwarnings("ignore")

from ultralytics import YOLO
print("✅ Libraries loaded successfully")

In [ ]:
import shutil
import random
from pathlib import Path

# =========================================
# CONFIG
# =========================================
SRC_DIR = Path(r"D:\Code\Hu_hong_đường\crack-1")
DST_DIR = Path(r"D:\Code\Hu_hong_đường\crack-2")

CLASSES_TO_BALANCE = {
    1: 3,   # spall → nhân 3 lần
    2: 6    # bt → nhân 6 lần (ít nhất)
}

# =========================================
# 1. COPY DATASET
# =========================================
def copy_dataset(src, dst):
    if dst.exists():
        print("⚠️ Folder đích đã tồn tại, xoá trước khi copy...")
        shutil.rmtree(dst)

    shutil.copytree(src, dst)
    print("✅ Đã copy dataset sang crack-2")

# =========================================
# 2. OVERSAMPLE CLASS
# =========================================
def oversample_split(split):
    img_dir = DST_DIR / split / "images"
    lbl_dir = DST_DIR / split / "labels"

    images = list(img_dir.glob("*.jpg"))

    for img_path in images:
        lbl_path = lbl_dir / (img_path.stem + ".txt")
        if not lbl_path.exists():
            continue

        with open(lbl_path, "r") as f:
            lines = f.readlines()

        # lấy class trong file label
        classes_in_img = [int(line.split()[0]) for line in lines if line.strip()]

        for cls, times in CLASSES_TO_BALANCE.items():
            if cls in classes_in_img:
                for i in range(times):
                    new_name = f"{img_path.stem}_aug_{cls}_{i}"

                    new_img = img_dir / f"{new_name}.jpg"
                    new_lbl = lbl_dir / f"{new_name}.txt"

                    shutil.copy(img_path, new_img)
                    shutil.copy(lbl_path, new_lbl)

    print(f"✅ Oversample xong split: {split}")

# =========================================
# RUN
# =========================================
copy_dataset(SRC_DIR, DST_DIR)

for split in ["train"]:   # chỉ oversample train
    oversample_split(split)

print("🎉 DONE tạo dataset crack-2!")

# Configuration

In [ ]:
# Cell 3: Configuration
DATA_DIR    = Path(r"D:\Code\Hu_hong_đường\crack-2")
IMG_SIZE    = 640           # Resize target (stretch to 640x640)
EPOCHS      = 100
BATCH       = 16
MODEL_NAME  = "yolov8n-seg.pt"   # nano-seg, thay bằng yolov8s-seg.pt nếu muốn lớn hơn
PROJECT     = str(DATA_DIR / "runs")
RUN_NAME    = "crack_seg_v1"

# Sub-directories (Roboflow structure)
TRAIN_IMG   = DATA_DIR / "train" / "images"
TRAIN_LBL   = DATA_DIR / "train" / "labels"
VALID_IMG   = DATA_DIR / "valid" / "images"
VALID_LBL   = DATA_DIR / "valid" / "labels"
TEST_IMG    = DATA_DIR / "test"  / "images"
TEST_LBL    = DATA_DIR / "test"  / "labels"

print(f"📁 Dataset root : {DATA_DIR}")
print(f"🖼️  Image size   : {IMG_SIZE}x{IMG_SIZE} (stretch)")
print(f"🔁 Epochs       : {EPOCHS}")
print(f"📦 Batch size   : {BATCH}")
print(f"🧠 Model        : {MODEL_NAME}")

##  Verify directory structure & data.yaml

In [ ]:
# Cell 4: Verify directory structure & data.yaml
def check_dir(path, label):
    exists = path.exists()
    count  = len(list(path.glob("*"))) if exists else 0
    status = "✅" if exists else "❌"
    print(f"{status} {label}: {path}  [{count} files]")
    return exists

print("=" * 60)
print("📂 Directory check")
print("=" * 60)
check_dir(TRAIN_IMG, "Train images")
check_dir(TRAIN_LBL, "Train labels")
check_dir(VALID_IMG, "Valid images")
check_dir(VALID_LBL, "Valid labels")
check_dir(TEST_IMG,  "Test  images")
check_dir(TEST_LBL,  "Test  labels")

# Locate data.yaml
yaml_path = DATA_DIR / "data.yaml"
if yaml_path.exists():
    with open(yaml_path, "r") as f:
        cfg = yaml.safe_load(f)
    print(f"\n✅ data.yaml found")
    print(f"   Classes ({cfg.get('nc', '?')}): {cfg.get('names', [])}")
    CLASSES = cfg.get("names", [])
    NC      = cfg.get("nc", len(CLASSES))
else:
    print("❌ data.yaml NOT found – please check dataset path")
    CLASSES, NC = [], 0

## Preprocessing — Auto-Orient + Stretch to 640x640

In [ ]:
# Cell 5: Preprocessing — Auto-Orient + Stretch to 640x640
def preprocess_image(img_path: Path, size: int = 640) -> np.ndarray:
    """
    Auto-orient (EXIF) then stretch to size x size.
    Returns BGR numpy array.
    """
    pil_img  = Image.open(img_path)
    pil_img  = ImageOps.exif_transpose(pil_img)        # Auto-Orient
    pil_img  = pil_img.convert("RGB")
    pil_img  = pil_img.resize((size, size), Image.BILINEAR)  # Stretch (no padding)
    return cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)

# Quick test on first train image
sample_imgs = sorted(TRAIN_IMG.glob("*.jpg"))[:1] + sorted(TRAIN_IMG.glob("*.png"))[:1]
if sample_imgs:
    raw   = cv2.imread(str(sample_imgs[0]))
    proc  = preprocess_image(sample_imgs[0])
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(raw,  cv2.COLOR_BGR2RGB)); axes[0].set_title(f"Original {raw.shape[1]}×{raw.shape[0]}")
    axes[1].imshow(cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)); axes[1].set_title(f"Preprocessed {proc.shape[1]}×{proc.shape[0]}")
    for ax in axes: ax.axis("off")
    plt.suptitle("Auto-Orient → Stretch 640×640", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
    print("✅ Preprocessing test passed")
else:
    print("⚠️  No sample images found for test")

## Visualize labels overlaid on images (segmentation polygons)

In [ ]:
# Cell 6: Visualize labels overlaid on images (segmentation polygons)
COLORS = plt.cm.get_cmap("tab10", max(NC, 10))

def draw_seg_labels(img_path: Path, lbl_path: Path, classes: list, size: int = 640):
    img = preprocess_image(img_path, size)
    h, w = img.shape[:2]
    overlay = img.copy()

    if not lbl_path.exists():
        print(f"⚠️  Label not found: {lbl_path}")
        return img

    with open(lbl_path, "r") as f:
        lines = f.read().strip().splitlines()

    for line in lines:
        parts = list(map(float, line.split()))
        cls_id = int(parts[0])
        coords = parts[1:]  # x1 y1 x2 y2 ... (normalized polygon)
        pts = np.array(coords).reshape(-1, 2)
        pts[:, 0] *= w; pts[:, 1] *= h
        pts = pts.astype(np.int32)
        color = tuple(int(c * 255) for c in COLORS(cls_id)[:3])[::-1]  # BGR
        cv2.fillPoly(overlay, [pts], color)
        cv2.polylines(img, [pts], True, color, 2)
        label_txt = classes[cls_id] if cls_id < len(classes) else str(cls_id)
        if len(pts): cv2.putText(img, label_txt, tuple(pts[0]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    combined = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
    return combined

# Show 6 random labeled samples
import random
all_train_imgs = sorted(TRAIN_IMG.glob("*.jpg")) + sorted(TRAIN_IMG.glob("*.png"))
samples = random.sample(all_train_imgs, min(6, len(all_train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, img_p in enumerate(samples):
    lbl_p  = TRAIN_LBL / (img_p.stem + ".txt")
    vis    = draw_seg_labels(img_p, lbl_p, CLASSES)
    axes[i].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(img_p.name, fontsize=8)
    axes[i].axis("off")
for j in range(i+1, len(axes)): axes[j].axis("off")
plt.suptitle("🏷️ Label Check – Segmentation Masks on Images", fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

## Dataset distribution analysis & charts — FIXED

In [ ]:
# Cell 7: Dataset distribution analysis & charts — FIXED

def count_labels(label_dir: Path):
    cls_counter = Counter()
    img_counter = 0
    for lbl_file in label_dir.glob("*.txt"):
        img_counter += 1
        for line in lbl_file.read_text().strip().splitlines():
            if line.strip():
                cls_id = int(line.split()[0])
                cls_counter[cls_id] += 1
    return img_counter, cls_counter

splits = {"Train": TRAIN_LBL, "Valid": VALID_LBL, "Test": TEST_LBL}
stats  = {}
for split, lbl_dir in splits.items():
    n_img, cls_cnt = count_labels(lbl_dir)
    stats[split] = {"images": n_img, "labels": cls_cnt}
    print(f"{split}: {n_img} images | {dict(cls_cnt)}")

# ── Xác định tất cả class_ids từ TẤT CẢ splits (không chỉ train) ──
all_cls_ids = sorted(set(
    k for s in stats.values() for k in s["labels"].keys()
))
cls_names = [CLASSES[k] if k < len(CLASSES) else f"cls_{k}" for k in all_cls_ids]
n_cls = len(all_cls_ids)

# ── Figure 1: Image count per split + pie + label distribution ────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

split_names  = list(stats.keys())
split_counts = [stats[s]["images"] for s in split_names]

# Bar: images per split
bars = axes[0].bar(split_names, split_counts,
                   color=["#4CAF50","#2196F3","#FF9800"], edgecolor="black", width=0.5)
axes[0].bar_label(bars, padding=3, fontsize=11, fontweight="bold")
axes[0].set_title("📊 Image Count per Split", fontweight="bold")
axes[0].set_ylabel("Number of Images")
axes[0].set_ylim(0, max(split_counts) * 1.2)
axes[0].grid(axis="y", linestyle="--", alpha=0.5)

# Pie: split proportion
axes[1].pie(split_counts, labels=split_names, autopct="%1.1f%%",
            colors=["#4CAF50","#2196F3","#FF9800"], startangle=90,
            wedgeprops={"edgecolor":"white","linewidth":2})
axes[1].set_title("🥧 Dataset Split Proportion", fontweight="bold")

# Bar: label count per class in Train only
train_cls = stats["Train"]["labels"]
train_vals = [train_cls.get(k, 0) for k in all_cls_ids]  # dùng all_cls_ids
colors_cls = [plt.cm.get_cmap("tab10")(i / max(n_cls, 1)) for i in range(n_cls)]
bars2 = axes[2].bar(cls_names, train_vals, color=colors_cls, edgecolor="black")
axes[2].bar_label(bars2, padding=3, fontsize=11, fontweight="bold")
axes[2].set_title("🏷️ Annotation Count per Class (Train)", fontweight="bold")
axes[2].set_ylabel("Number of Annotations")
axes[2].set_xticklabels(cls_names, rotation=20, ha="right")
axes[2].grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Dataset Distribution Overview", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout(); plt.show()

# ── Figure 2: Per-class distribution across all splits ────────────────────
fig2, ax2 = plt.subplots(figsize=(max(8, n_cls * 2), 5))
x = np.arange(n_cls)
width = 0.25

for i, (split, color) in enumerate(zip(split_names, ["#4CAF50","#2196F3","#FF9800"])):
    # ✅ FIX: dùng all_cls_ids để đảm bảo vals cùng độ dài với x
    vals = [stats[split]["labels"].get(k, 0) for k in all_cls_ids]
    ax2.bar(x + i * width, vals, width, label=split, color=color, edgecolor="black")

ax2.set_xticks(x + width)
ax2.set_xticklabels(cls_names, rotation=20, ha="right")
ax2.set_title("📈 Annotation Distribution per Class & Split", fontweight="bold")
ax2.set_ylabel("Annotation Count")
ax2.legend()
ax2.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout(); plt.show()
print("✅ Distribution charts rendered")

## Ensure data.yaml is correct for segmentation

In [ ]:
# Cell 8: Ensure data.yaml is correct for segmentation
import shutil

yaml_content = {
    "path" : str(DATA_DIR),
    "train": "train/images",
    "val"  : "valid/images",
    "test" : "test/images",
    "nc"   : NC,
    "names": CLASSES
}

yaml_out = DATA_DIR / "data.yaml"
with open(yaml_out, "w") as f:
    yaml.dump(yaml_content, f, default_flow_style=False, allow_unicode=True)
print(f"✅ data.yaml written → {yaml_out}")
print(yaml_content)

In [ ]:
# Cell 9: Load YOLOv8-seg model and train

model = YOLO(MODEL_NAME)   # downloads yolov8n-seg.pt if not present
print(f"✅ Model loaded: {MODEL_NAME}")

results = model.train(
    data      = str(DATA_DIR / "data.yaml"),
    epochs    = EPOCHS,
    imgsz     = IMG_SIZE,           # 640
    batch     = BATCH,
    rect      = False,              # stretch (no rect padding) → matches your param
    augment   = False,              # No augmentations (as specified)
    cache     = False,
    workers   = 4,
    project   = PROJECT,
    name      = RUN_NAME,
    exist_ok  = True,
    save      = True,
    plots     = True,
    verbose   = True,
    # Optimizer defaults (v1 settings)
    optimizer = "SGD",
    lr0       = 0.01,
    lrf       = 0.01,
    momentum  = 0.937,
    weight_decay = 0.0005,
    warmup_epochs= 3,
    cos_lr    = False,
)
print("🎉 Training complete!")

In [ ]:
# =========================================
# 1. IMPORT
# =========================================
from ultralytics import YOLO
from pathlib import Path

# =========================================
# 2. CONFIG
# =========================================
DATA_DIR    = Path(r"D:\Code\Hu_hong_đường\crack-2")

MODEL_NAME  = "yolov8s-seg.pt"   # 🔥 upgrade từ n → s
PROJECT     = str(DATA_DIR / "runs")
RUN_NAME    = "crack_seg_v2_opt"


# =========================================
# 3. LOAD MODEL
# =========================================
model = YOLO(MODEL_NAME)
print("✅ Model loaded!")

# =========================================
# 4. TRAIN
# =========================================
results = model.train(
    data=str(DATA_DIR / "data.yaml"),

    # ===== core =====
    epochs=150,         # 🔥 đủ học, tránh overfit
    imgsz=832,          # 🔥 đủ chi tiết, không quá nặng
    batch=8,

    # ===== model =====
    model="yolov8s-seg.pt",

    # ===== optimizer =====
    optimizer="AdamW",
    lr0=0.0005,
    cos_lr=True,
    warmup_epochs=5,

    # ===== augment =====
    augment=True,
    mosaic=0.5,
    mixup=0.0,

    # ===== imbalance =====
    cls=1.0,

    # ===== regularization =====
    dropout=0.1,        # 🔥 chống overfit (rất hợp case bạn)

    # ===== system =====
    amp=True,
    workers=2,
)

print("🎉 TRAINING DONE!")